# AIMO Progress Prize 3 — TIR Baseline Pipeline

**Approach**: Tool-Integrated Reasoning (TIR) + Self-Correction + Majority Voting (Self-Consistency)

- Model: `DeepSeek-R1-Distill-Qwen-14B-AWQ` from `/kaggle/input/<dataset-slug>`
- Engine: vLLM (offline `LLM` class)
- Inference: generate `NUM_SAMPLES` independent samples → majority vote
- TIR: LLM writes Python → sandbox executes → output fed back → self-correct (up to `MAX_TIR_ROUNDS`)

In [ ]:
# ============================================================
# Cell 0: Install dependencies (vLLM not pre-installed on Kaggle)
# ============================================================
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

_pip('vllm')
print('vllm installed.')


In [ ]:
# ============================================================
# Cell 1: Imports & Environment Detection
# ============================================================
import os
import re
import sys
import time
import tempfile
import subprocess
from pathlib import Path
from collections import Counter
from typing import List, Optional

import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local (mock mode)'}")
print(f"Python      : {sys.version.split()[0]}")


In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

# --- Model path (loaded from Kaggle Dataset) ---
# Upload weights via: kaggle datasets create -p ./model-weights
MODEL_DATASET   = 'deepseek-r1-distill-qwen-14b-awq'          # dataset slug
MODEL_PATH      = f'/kaggle/input/{MODEL_DATASET}'             # full path on Kaggle
QUANTIZATION    = 'awq'    # set None for bf16 (requires more VRAM)

# --- vLLM engine ---
GPU_MEMORY_UTIL = 0.92     # fraction of H100 80 GB VRAM
MAX_MODEL_LEN   = 32768    # max context length (prompt + generation)

# --- Sampling ---
NUM_SAMPLES     = 16 if IS_KAGGLE else 4   # maj@N; lower for local mock test
TEMPERATURE     = 0.6
TOP_P           = 0.95
MAX_NEW_TOKENS  = 16384    # per sample

# --- TIR self-correction ---
MAX_TIR_ROUNDS  = 3        # max (code exec -> continue) iterations per sample
CODE_TIMEOUT    = 30       # seconds per code block execution

# --- I/O paths ---
if IS_KAGGLE:
    TEST_CSV = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv'
else:
    TEST_CSV = 'dummy_test.csv'
OUTPUT_CSV = 'submission.csv'

print('Configuration:')
for k, v in {
    'MODEL_PATH': MODEL_PATH, 'NUM_SAMPLES': NUM_SAMPLES,
    'MAX_TIR_ROUNDS': MAX_TIR_ROUNDS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS,
    'TEMPERATURE': TEMPERATURE, 'CODE_TIMEOUT': CODE_TIMEOUT,
}.items():
    print(f'  {k:20s}: {v}')


In [ ]:
# ============================================================
# Cell 3: Python Sandbox
# ============================================================

def execute_python(code: str, timeout: int = CODE_TIMEOUT) -> str:
    """
    Execute Python code in an isolated subprocess.
    Returns combined stdout + stderr, truncated to 2000 chars.
    """
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='.py', delete=False, encoding='utf-8'
    ) as f:
        f.write(code)
        tmp_path = f.name

    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        output = proc.stdout + proc.stderr
        if not output.strip():
            output = '(no output)'
    except subprocess.TimeoutExpired:
        output = f'TimeoutError: execution exceeded {timeout}s'
    except Exception as exc:
        output = f'ExecutionError: {exc}'
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass

    if len(output) > 2000:
        output = output[:1500] + '\n...[truncated]...\n' + output[-500:]
    return output.rstrip()


# Sanity check
_test = execute_python('ans = sum(range(1, 101))\nprint(ans)')
assert _test.strip() == '5050', f'Sandbox test failed: {_test!r}'
print(f'Sandbox OK  -> {_test}')


In [ ]:
# ============================================================
# Cell 4: Prompt Templates & Regex Helpers
# ============================================================

SYSTEM_PROMPT = (
    'You are an expert mathematician solving competition math problems.\n'
    'Reason step by step. You may write Python code to assist with calculations.\n'
    '\n'
    'Wrap Python code in triple backticks:\n'
    '```python\n'
    '# your code here\n'
    '```\n'
    'The code will be executed and the output shown to you.\n'
    'Use code for: arithmetic, modular arithmetic, combinatorics, exhaustive search, verification.\n'
    '\n'
    'After reasoning, state your final answer EXACTLY as:\n'
    'The answer is \\boxed{<integer>}'
)


def build_initial_prompt(problem: str) -> str:
    return f'{SYSTEM_PROMPT}\n\nProblem: {problem}\n\nSolution:'


_CODE_RE  = re.compile(r'```python\s*(.*?)```', re.DOTALL)
_BOXED_RE = re.compile(r'\\boxed\{([^}]+)\}')


def extract_code_blocks(text: str) -> List[str]:
    return [b.strip() for b in _CODE_RE.findall(text)]


def extract_final_answer(text: str) -> Optional[str]:
    matches = _BOXED_RE.findall(text)
    return matches[-1].strip() if matches else None


assert extract_final_answer('The answer is \\boxed{42}') == '42'
assert extract_code_blocks('```python\nprint(1)\n```') == ['print(1)']
print('Prompt helpers OK')


In [ ]:
# ============================================================
# Cell 5: Model Loading  (vLLM on Kaggle / MockLLM locally)
# ============================================================

if IS_KAGGLE:
    from vllm import LLM, SamplingParams  # type: ignore[import]

    print(f'Loading model from {MODEL_PATH} ...')
    llm = LLM(
        model=MODEL_PATH,
        quantization=QUANTIZATION if QUANTIZATION else None,
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        max_model_len=MAX_MODEL_LEN,
        dtype='auto',
        trust_remote_code=True,
        enforce_eager=False,
    )
    print('Model loaded.')

else:
    print('Local mode: using MockLLM')

    class _MockCompletionOutput:
        def __init__(self, text: str):
            self.text = text

    class _MockRequestOutput:
        def __init__(self, texts: List[str]):
            self.outputs = [_MockCompletionOutput(t) for t in texts]

    # Four branches to exercise all pipeline paths:
    #   0: code + boxed  (happy path)
    #   1: code, no boxed (triggers self-correction)
    #   2: direct boxed, no code
    #   3: continuation response (used during self-correction)
    _MOCK_RESPONSES = [
        (
            'Let me compute this.\n'
            '```python\n'
            'total = sum(range(1, 101))\n'
            'print(total)\n'
            '```\n'
            'The sum is 5050.\n'
            'The answer is \\boxed{5050}'
        ),
        (
            'Let me write code.\n'
            '```python\n'
            'n = 100\n'
            'print(n * (n + 1) // 2)\n'
            '```'
        ),
        (
            'Using n(n+1)/2 with n=100: 5050.\n'
            'The answer is \\boxed{5050}'
        ),
        (
            'The execution output confirms the answer.\n'
            'The answer is \\boxed{5050}'
        ),
    ]

    class MockSamplingParams:
        def __init__(self, temperature=0.6, top_p=0.95, max_tokens=4096, n=1):
            self.temperature = temperature
            self.top_p = top_p
            self.max_tokens = max_tokens
            self.n = n

    class MockLLM:
        _call_count = 0

        def generate(self, prompts: List[str], sampling_params) -> List[_MockRequestOutput]:
            results = []
            for _ in prompts:
                texts = [
                    _MOCK_RESPONSES[(MockLLM._call_count + i) % len(_MOCK_RESPONSES)]
                    for i in range(sampling_params.n)
                ]
                MockLLM._call_count += sampling_params.n
                results.append(_MockRequestOutput(texts))
            return results

    llm = MockLLM()
    SamplingParams = MockSamplingParams  # type: ignore[assignment,misc]
    print('MockLLM ready.')


In [ ]:
# ============================================================
# Cell 6: TIR Inference with Self-Correction
# ============================================================

def _run_tir_continuation(
    conversation: str,
    llm_instance,
    max_rounds: int = MAX_TIR_ROUNDS,
) -> str:
    """
    Given a partial conversation ending without a \\boxed{} answer,
    execute code blocks and continue generating until an answer appears
    or max_rounds is exhausted.
    Returns the final answer string, or '' if not found.
    """
    sp = SamplingParams(
        temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS, n=1
    )
    latest_response = conversation  # first entry: entire conversation is 'latest'

    for _ in range(max_rounds):
        code_blocks = extract_code_blocks(latest_response)
        if not code_blocks:
            break

        code_output = execute_python(code_blocks[-1])
        conversation += (
            f'\n\nExecution output:\n```\n{code_output}\n```\n\n'
            'Continue your solution based on the output above:'
        )

        cont_out = llm_instance.generate([conversation], sp)
        latest_response = cont_out[0].outputs[0].text
        conversation += latest_response

        answer = extract_final_answer(latest_response)
        if answer is not None:
            return answer

    return extract_final_answer(conversation) or ''


def tir_batch(
    problem: str,
    llm_instance,
    num_samples: int = NUM_SAMPLES,
) -> List[str]:
    """
    Generate num_samples independent TIR solutions to problem.

    1. Batch-generate all samples in one vLLM call (maximises GPU use).
    2. Samples with code but no boxed answer -> self-correction loop.
    3. Samples with no code and no answer   -> boxed-answer nudge.
    4. Return list of answer strings.        ('': failed samples)
    """
    initial_prompt = build_initial_prompt(problem)
    sp_batch = SamplingParams(
        temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS, n=num_samples
    )
    batch_out = llm_instance.generate([initial_prompt], sp_batch)

    answers: List[str] = []
    for sample_out in batch_out[0].outputs:
        response_text = sample_out.text

        answer = extract_final_answer(response_text)
        if answer is not None:
            answers.append(answer)
            continue

        if extract_code_blocks(response_text):
            answer = _run_tir_continuation(
                initial_prompt + response_text, llm_instance
            )
        else:
            nudge = (
                initial_prompt + response_text
                + '\n\nPlease state the final integer answer using \\boxed{<integer>}:'
            )
            sp_nudge = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=256, n=1)
            nudge_out = llm_instance.generate([nudge], sp_nudge)
            answer = extract_final_answer(nudge_out[0].outputs[0].text) or ''

        answers.append(answer)

    return answers


In [ ]:
# ============================================================
# Cell 7: Self-Consistency (Majority Voting)
# ============================================================

def majority_vote(answers: List[str]) -> str:
    """Most frequent non-empty answer; '0' as fallback."""
    valid = [a for a in answers if a.strip()]
    if not valid:
        return '0'
    return Counter(valid).most_common(1)[0][0]


def solve_problem(
    problem: str,
    llm_instance,
    num_samples: int = NUM_SAMPLES,
) -> dict:
    """Full pipeline for one problem: TIR batch -> majority vote."""
    all_answers = tir_batch(problem, llm_instance, num_samples)
    final_answer = majority_vote(all_answers)
    vote_dist = Counter(a for a in all_answers if a)
    return {
        'answer': final_answer,
        'all_answers': all_answers,
        'vote_distribution': dict(vote_dist),
        'num_valid': sum(1 for a in all_answers if a),
    }


print('Voting helpers OK')


In [ ]:
# ============================================================
# Cell 8: Main Pipeline
# ============================================================

def run_pipeline(
    llm_instance,
    test_csv: str = TEST_CSV,
    output_csv: str = OUTPUT_CSV,
    num_samples: int = NUM_SAMPLES,
) -> pd.DataFrame:
    """
    Read test.csv  (columns: id, problem)
    Write output_csv (columns: id, answer)
    """
    df = pd.read_csv(test_csv)
    assert {'id', 'problem'}.issubset(df.columns), (
        f'Expected id+problem columns, got {list(df.columns)}'
    )

    rows = []
    total = len(df)
    t_start = time.time()

    for loop_idx, (_, row) in enumerate(df.iterrows()):
        pid   = row['id']
        ptext = row['problem']

        print(f"\n{'=' * 60}")
        print(f'[{loop_idx + 1}/{total}] id={pid}')
        print(f"Problem : {ptext[:120]}{'...' if len(ptext) > 120 else ''}")

        t0     = time.time()
        result = solve_problem(ptext, llm_instance, num_samples)
        elapsed = time.time() - t0

        print(f"Answer  : {result['answer']}")
        print(f"Votes   : {result['vote_distribution']}")
        print(f"Valid   : {result['num_valid']}/{num_samples}")
        print(f'Time    : {elapsed:.1f}s')

        rows.append({'id': pid, 'answer': result['answer']})

    total_elapsed = time.time() - t_start
    print(f"\n{'=' * 60}")
    print(f'Pipeline done: {total} problems in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')

    sub_df = pd.DataFrame(rows)
    sub_df['answer'] = (
        pd.to_numeric(sub_df['answer'], errors='coerce')
        .fillna(0)
        .astype(int)
    )
    sub_df.to_csv(output_csv, index=False)
    print(f'Saved   : {output_csv}')
    print(sub_df.to_string(index=False))
    return sub_df


In [ ]:
# ============================================================
# Cell 9: Local Dummy Test  (skipped on Kaggle)
# ============================================================

if not IS_KAGGLE:
    print('>>> Running local pipeline test with dummy data <<<')

    dummy_problems = [
        {'id': 1, 'problem': 'What is the sum of the integers from 1 to 100?'},
        {'id': 2, 'problem': 'Find the remainder when 2^10 is divided by 7.'},
        {'id': 3, 'problem': 'How many prime numbers are less than 20?'},
    ]
    pd.DataFrame(dummy_problems).to_csv(TEST_CSV, index=False)
    print(f'Dummy test.csv -> {TEST_CSV}')

    submission = run_pipeline(llm, test_csv=TEST_CSV, output_csv=OUTPUT_CSV)

    result_df = pd.read_csv(OUTPUT_CSV)
    assert list(result_df.columns) == ['id', 'answer'], f'Bad columns: {list(result_df.columns)}'
    assert len(result_df) == len(dummy_problems), 'Row count mismatch'
    assert pd.api.types.is_integer_dtype(result_df['answer']), (
        f"'answer' is not integer dtype: {result_df['answer'].dtype}"
    )
    print('\n>>> All assertions passed. submission.csv format is valid. <<<')


In [ ]:
# ============================================================
# Cell 10: Run on Kaggle (real test.csv)
# ============================================================

if IS_KAGGLE:
    submission = run_pipeline(llm, test_csv=TEST_CSV, output_csv=OUTPUT_CSV)
